# SpaceX Falcon 9 Landing Prediction
Isse Abdallah Mohamoud

Course snapshots reproduce historical lab scopes. Run collect_data.py to download inputs. The report separates 90 model records, 101 SQL records and 56 dashboard records. See collection_status.json for the blocked historical Wikipedia request.

In [ ]:
from pathlib import Path
import pandas as pd, json
R=Path('.')
df=pd.read_csv(R/'data/dataset_part_2.csv')
print(df.shape)
display(df.head())

In [ ]:
display(df.groupby('LaunchSite').Class.agg(['count','sum','mean']))
display(df.groupby('Orbit').Class.agg(['count','sum','mean']))

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
sns.scatterplot(data=df,x='FlightNumber',y='LaunchSite',hue='Class')
plt.show()

In [ ]:
sns.scatterplot(data=df,x='PayloadMass',y='LaunchSite',hue='Class')
plt.show()

In [ ]:
df.groupby('Orbit').Class.mean().plot.bar(ylabel='Landing success rate')
plt.show()

In [ ]:
sns.scatterplot(data=df,x='FlightNumber',y='Orbit',hue='Class')
plt.show()

In [ ]:
sns.scatterplot(data=df,x='PayloadMass',y='Orbit',hue='Class')
plt.show()

In [ ]:
df.assign(Year=pd.to_datetime(df.Date).dt.year).groupby('Year').Class.mean().plot(marker='o',ylabel='Landing success rate')
plt.show()

In [ ]:
import sqlite3
con=sqlite3.connect(':memory:')
sql_df=pd.read_csv(R/'data/Spacex.csv')
for col in sql_df.select_dtypes('object'): sql_df[col]=sql_df[col].str.strip()
sql_df.to_sql('SPACEXTBL',con,index=False)
results=json.loads((R/'analysis_results.json').read_text())
for title,query in results['queries'].items():
 print(title)
 display(pd.read_sql_query(query,con))

## Predictive analysis
Run `python analyze.py` to reproduce model tuning and export analysis_results.json. Scaling fits inside each training fold. A separate lab-style global-scaling comparison illustrates the difference in validation design. Source payload imputation and repeated booster histories remain limitations.

In [ ]:
results=json.loads((R/'analysis_results.json').read_text())
display(pd.DataFrame(results['results'])[['model','cv_accuracy','test_accuracy','precision','recall','f1']])
print('Selected by validation:',results['best_model'])
print('Confusion matrix:',next(r['confusion_matrix'] for r in results['results'] if r['model']==results['best_model']))